# UniXcoder + DFG Training

Fine-tunes `microsoft/unixcoder-base` with the same DFG-aware sparse attention mask used by GraphCodeBERT — treating UniXcoder as a drop-in encoder replacement.

**Split:** 90/10 train/test (no validation during training)  
**Output:** `saved_models_unixcoder_dfg/model_unixcoder_dfg.bin`

In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 8.6 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import logging
import random
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler, random_split
from torch.optim import AdamW
from transformers import (get_linear_schedule_with_warmup,
                          RobertaConfig, RobertaModel, AutoTokenizer)
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

# ─── CONFIGURATION ────────────────────────────────────────────────────────
class Args:
    output_dir         = "saved_models_unixcoder_dfg"
    model_name_or_path = "microsoft/unixcoder-base"

    # Data path
    train_file         = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"

    # Sequence lengths (same as GraphCodeBERT for fair comparison)
    code_length        = 384
    data_flow_length   = 128

    # Hyperparameters
    train_batch_size   = 16
    eval_batch_size    = 32
    learning_rate      = 2e-5
    max_grad_norm      = 1.0
    num_train_epochs   = 3
    seed               = 42

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = Args()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(args.seed)
print(f"Device : {args.device}")
print(f"Model  : {args.model_name_or_path}")
print(f"Input  : code_length={args.code_length}  dfg_length={args.data_flow_length}")


Device : cuda
Model  : microsoft/unixcoder-base
Input  : code_length=384  dfg_length=128


In [3]:
# ─── DFG-AWARE MODEL ─────────────────────────────────────────────────────
# UniXcoder uses the same RoBERTa-style internal API as GraphCodeBERT,
# so the custom sparse attention forward pass works without modification.
class Model(nn.Module):
    def __init__(self, encoder, config):
        super(Model, self).__init__()
        self.encoder    = encoder
        self.config     = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        # Convert boolean mask: 1 = attend, 0 = mask -> 0 / -10000
        extended_attention_mask = (1.0 - attn_mask) * -10000.0
        extended_attention_mask = extended_attention_mask.unsqueeze(1)
        # Shape: [Batch, 1, Seq, Seq]

        # Embed tokens with custom position IDs (DFG nodes use pos=0)
        embedding_output = self.encoder.embeddings(
            input_ids=input_ids,
            position_ids=p_ids
        )

        # Pass through transformer layers with the sparse mask
        encoder_outputs = self.encoder.encoder(
            embedding_output,
            attention_mask=extended_attention_mask,
            head_mask=[None] * self.config.num_hidden_layers
        )

        sequence_output = encoder_outputs[0]
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)

        if labels is not None:
            loss = CrossEntropyLoss()(logits, labels)
            return loss, prob
        return prob


In [4]:
# ─── DFG-AWARE DATASET ───────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args      = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length

        with open(file_path, 'r') as f:
            self.lines = f.readlines()

    def __len__(self):
        return len(self.lines)

    def _get_char_index(self, code_lines, coord):
        row, col = coord
        char_idx = 0
        for i in range(min(row, len(code_lines))):
            char_idx += len(code_lines[i])
        return char_idx + col

    def __getitem__(self, item):
        entry      = json.loads(self.lines[item])
        code       = entry.get('code', '')
        dfg        = entry.get('dfg', [])[:self.args.data_flow_length]
        label      = int(entry.get('label', 0)) if entry.get('label') is not None else 0
        code_lines = code.splitlines(keepends=True)

        # ── 1. Tokenize with offset mapping for DFG alignment ─────────────
        tokens_obj = self.tokenizer(
            code,
            max_length=self.args.code_length,
            truncation=True,
            padding='max_length',
            return_offsets_mapping=True
        )
        input_ids = tokens_obj['input_ids']
        offsets   = tokens_obj['offset_mapping']

        # ── 2. Map DFG nodes to code tokens ───────────────────────────────
        dfg_ids          = [self.tokenizer.unk_token_id] * len(dfg)
        node_to_token_map = {}
        pos_to_node_idx   = {}

        for node_idx, node_item in enumerate(dfg):
            try:
                start_pos = node_item[1][0]
                end_pos   = node_item[1][1]
                if len(start_pos) < 2 or len(end_pos) < 2:
                    node_to_token_map[node_idx] = []
                    continue
                pos_key = (start_pos[0], start_pos[1], end_pos[0], end_pos[1])
                pos_to_node_idx[pos_key] = node_idx

                char_start = self._get_char_index(code_lines, start_pos)
                char_end   = self._get_char_index(code_lines, end_pos)

                aligned = []
                for t_idx, (t_start, t_end) in enumerate(offsets):
                    if t_start == t_end:
                        continue
                    if (t_start >= char_start and t_end <= char_end) or \
                       (char_start >= t_start and char_end <= t_end):
                        aligned.append(t_idx)
                node_to_token_map[node_idx] = aligned
            except (IndexError, TypeError):
                node_to_token_map[node_idx] = []

        # ── 3. Build sparse attention mask ────────────────────────────────
        c_len    = self.args.code_length
        attn_mask = np.zeros((self.total_len, self.total_len), dtype=bool)

        # A. Code tokens attend to each other freely
        attn_mask[:c_len, :c_len] = True

        # B. DFG node <-> aligned code tokens (bidirectional anchor)
        # C. DFG node <-> parent nodes along data-flow edges (bidirectional)
        # D. Self-loop for every DFG node
        for node_idx, node_item in enumerate(dfg):
            abs_node = c_len + node_idx

            for t_idx in node_to_token_map.get(node_idx, []):
                attn_mask[abs_node, t_idx] = True
                attn_mask[t_idx, abs_node] = True

            try:
                for p_pos in node_item[4]:
                    if len(p_pos) < 2:
                        continue
                    p_key = (p_pos[0][0], p_pos[0][1], p_pos[1][0], p_pos[1][1])
                    if p_key in pos_to_node_idx:
                        abs_parent = c_len + pos_to_node_idx[p_key]
                        attn_mask[abs_node, abs_parent] = True
                        attn_mask[abs_parent, abs_node] = True
            except (IndexError, TypeError):
                pass

            attn_mask[abs_node, abs_node] = True

        # ── 4. Assemble final sequence ─────────────────────────────────────
        full_input_ids = input_ids + dfg_ids
        # Code tokens: sequential positions starting at 2 (0=pad, 1=special)
        # DFG nodes:   fixed position 0 (non-sequential signal)
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)

        pad = self.total_len - len(full_input_ids)
        if pad > 0:
            full_input_ids += [self.tokenizer.pad_token_id] * pad
            p_ids          += [1] * pad

        return {
            'input_ids': torch.tensor(full_input_ids, dtype=torch.long),
            'p_ids':     torch.tensor(p_ids,          dtype=torch.long),
            'attn_mask': torch.tensor(attn_mask,      dtype=torch.float),
            'label':     torch.tensor(label,          dtype=torch.long)
        }


In [5]:
# ─── TOKENIZER + 90/10 TRAIN/TEST SPLIT ─────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=True)
full_dataset = TextDataset(tokenizer, args, args.train_file)

total      = len(full_dataset)
train_size = int(0.9 * total)
test_size  = total - train_size

generator = torch.Generator().manual_seed(args.seed)
train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size], generator=generator
)

print(f"Dataset split:")
print(f"  Total : {total:,}")
print(f"  Train : {train_size:,} (90%)")
print(f"  Test  : {test_size:,} (10%)")


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/vocab.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/vocab.json "HTTP/1.1 200 OK"


...

merges.txt: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Dataset split:
  Total : 199,960
  Train : 179,964 (90%)
  Test  : 19,996 (10%)


In [6]:
# ─── TRAINING LOOP ───────────────────────────────────────────────────────
def train(model, train_dataset, args):
    train_dataloader = DataLoader(
        train_dataset,
        sampler=RandomSampler(train_dataset),
        batch_size=args.train_batch_size,
        num_workers=2,
        pin_memory=True
    )

    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=len(train_dataloader) * args.num_train_epochs
    )
    scaler = GradScaler()

    if not os.path.exists(args.output_dir):
        os.makedirs(args.output_dir)

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0
        bar = tqdm(train_dataloader, desc=f"Epoch {epoch}")
        for step, batch in enumerate(bar):
            optimizer.zero_grad()
            with autocast():
                loss, _ = model(
                    input_ids=batch['input_ids'].to(args.device),
                    p_ids=batch['p_ids'].to(args.device),
                    attn_mask=batch['attn_mask'].to(args.device),
                    labels=batch['label'].to(args.device)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss += loss.item()
            bar.set_postfix(loss=tr_loss / (step + 1))

        logger.info(f"Epoch {epoch} | Loss: {tr_loss / len(train_dataloader):.4f}")

    # Save final model after all epochs
    save_path = os.path.join(args.output_dir, "model_unixcoder_dfg.bin")
    torch.save(model.state_dict(), save_path)
    logger.info(f"Saved final model to {save_path}")


def evaluate(model, dataset, args, tag="Test"):
    dataloader = DataLoader(
        dataset,
        sampler=SequentialSampler(dataset),
        batch_size=args.eval_batch_size,
        num_workers=2,
        pin_memory=True
    )
    model.eval()
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {tag}"):
            probs = model(
                input_ids=batch['input_ids'].to(args.device),
                p_ids=batch['p_ids'].to(args.device),
                attn_mask=batch['attn_mask'].to(args.device)
            )
            all_probs.append(probs.cpu().numpy())
            all_labels.extend(batch['label'].cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.argmax(all_probs, axis=-1)

    acc     = accuracy_score(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs[:, 1])
    pr_auc  = average_precision_score(all_labels, all_probs[:, 1])
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

    print("\n" + "=" * 40)
    print(f"FINAL TEST RESULTS (UniXcoder + DFG)")
    print("=" * 40)
    print(f"Accuracy : {acc:.4%}")
    print(f"ROC-AUC  : {roc_auc:.4f}")
    print(f"PR-AUC   : {pr_auc:.4f}")
    print(f"FN Count : {fn}  (missed vulnerabilities)")
    print(f"FP Count : {fp}  (false alarms)")
    print("-" * 40)
    print(classification_report(all_labels, all_preds, target_names=['Safe', 'Vuln'], digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return acc, all_probs, np.array(all_labels)


In [7]:
# ─── MAIN ────────────────────────────────────────────────────────────────
# Initialization
config  = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model   = Model(encoder, config)
model.to(args.device)

# Train on full 90% split (no validation during training)
train(model, train_dataset, args)

# Load final checkpoint and evaluate once on the held-out 10% test set
model.load_state_dict(
    torch.load(os.path.join(args.output_dir, "model_unixcoder_dfg.bin"))
)
acc, probs, labels = evaluate(model, test_dataset, args, tag="UniXcoder+DFG")

# Save raw outputs for downstream analysis
np.save("/kaggle/working/test_probs_unixcoder_dfg.npy", probs)
np.save("/kaggle/working/test_labels_unixcoder_dfg.npy", labels)

# Write summary metrics to text file
preds   = np.argmax(probs, axis=-1)
roc_auc = roc_auc_score(labels, probs[:, 1])
pr_auc  = average_precision_score(labels, probs[:, 1])
tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

out_path = "/kaggle/working/unixcoder_dfg_results.txt"
with open(out_path, "w") as f:
    f.write(f"Model        : UniXcoder + DFG sparse attention\n")
    f.write(f"Split        : 90/10 train/test (no validation)\n")
    f.write(f"Epochs       : {args.num_train_epochs}\n")
    f.write(f"Accuracy     : {acc:.4%}\n")
    f.write(f"ROC-AUC      : {roc_auc:.4f}\n")
    f.write(f"PR-AUC       : {pr_auc:.4f}\n")
    f.write(f"FN           : {fn}\n")
    f.write(f"FP           : {fp}\n")
print(f"Saved results to {out_path}")


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/commits/main "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/discussions?p=0 "HTTP/1.1 200 OK"
RobertaModel LOAD REPORT from: microsoft/unixcoder-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/commits/refs%2Fpr%2F8 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/refs%2Fpr%2F8/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/refs%2Fpr%2F8/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/xet-read-token/558aa506226b13a7e83f66cb38c53e61b9603eac "HTTP/1

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

/tmp/ipykernel_24/3187414951.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()

Epoch 0:   0%|          | 0/11248 [00:00<?, ?it/s]/tmp/ipykernel_24/3187414951.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():

Epoch 0: 100%|██████████| 11248/11248 [2:30:28<00:00,  1.25it/s, loss=0.299]
INFO:__main__:Epoch 0 | Loss: 0.2991
Epoch 1: 100%|██████████| 11248/11248 [2:30:50<00:00,  1.24it/s, loss=0.237]
INFO:__main__:Epoch 1 | Loss: 0.2368
Epoch 2: 100%|██████████| 11248/11248 [2:30:44<00:00,  1.24it/s, loss=0.192]
INFO:__main__:Epoch 2 | Loss: 0.1918
INFO:__main__:Saved final model to saved_models_unixcoder_dfg/model_unixcoder_dfg.bin
Evaluating UniXcoder+DFG: 100%|██████████| 625/625 [05:06<00:00,  2.04it/s]


FINAL TEST RESULTS (UniXcoder + DFG)
Accuracy : 89.4029%
ROC-AUC  : 0.9651
PR-AUC   : 0.9657
FN Count : 1043  (missed vulnerabilities)
FP Count : 1076  (false alarms)
----------------------------------------
              precision    recall  f1-score   support

        Safe     0.8963    0.8934    0.8949     10095
        Vuln     0.8917    0.8947    0.8932      9901

    accuracy                         0.8940     19996
   macro avg     0.8940    0.8940    0.8940     19996
weighted avg     0.8940    0.8940    0.8940     19996

Confusion Matrix:
[[9019 1076]
 [1043 8858]]
Saved results to /kaggle/working/unixcoder_dfg_results.txt
